# 01 · 1990년대 차트 선별
원본: Bill_Board/BillBoard_Chart_analysis.ipynb

차트 수집 이후의 곡 선별과 작업 배분 단계입니다. 원곡 목록은 포함하지 않습니다. pandas의 제거된 append 사용을 직접 행 선택으로 교체했습니다.

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("차트 CSV 하나를 업로드하세요.")
file_path = next(iter(uploaded))
BB = pd.read_csv(file_path)
required = {"rank", "date", "song", "artist", "last_week", "weeks_on_chart"}
if not required.issubset(BB.columns):
    raise ValueError(f"필요한 컬럼: {sorted(required)}")

In [ ]:
bb = BB[['rank', 'date', 'song', 'artist', 'last_week', 'weeks_on_chart']]
bb90 = bb[bb['date'].astype(str).str.contains('199', na=False)].fillna(0)
bb90 = bb90.drop(columns=['last_week'])
b90_sorted = bb90.sort_values(by='weeks_on_chart', ascending=False)
bb90_sorted_droped = b90_sorted.drop_duplicates(subset='song', ignore_index=True)
if len(bb90_sorted_droped) < 300:
    raise ValueError('당시 배분 규칙에는 선별된 곡이 최소 300개 필요합니다.')

## 남아 있는 배분 단계
0 기준 120–239번을 세 명에게 40개씩, 240–299번을 재명에게 배정합니다. 곡명만으로 중복을 제거하는 원본 기준을 유지합니다.

In [ ]:
Haewan = bb90_sorted_droped.iloc[120:240:3].reset_index(drop=True)
Boram = bb90_sorted_droped.iloc[121:240:3].reset_index(drop=True)
Dongjin = bb90_sorted_droped.iloc[122:240:3].reset_index(drop=True)
Jaemyung = bb90_sorted_droped.iloc[240:300].reset_index(drop=True)
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
for name, rows in [('Haewan', Haewan), ('Boram', Boram), ('Dongjin', Dongjin), ('Jaemyung', Jaemyung)]:
    rows.to_csv(output_dir / f'{name}1.csv')
    print(name, len(rows))